In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path
import whisper
import lightning_scripts.whisper_transfer_module as whisper_module

In [2]:
torch.cuda.is_available()

True

In [ ]:
## init config. Will be yaml eventually, but start as dict 
config = {}
# Overwrite config for linear classifier 
config['num_workers'] = 4
config['hparas'] = {}
config['data'] = {}
config['hparas']['batch_size'] = 192
config['hparas']['optimizer'] = "AdamW"
config['hparas']['lr'] = 0.0005
config['data']['eval_max'] = 1
config['model'] = {}
config['model']['arch_kwargs'] = {} 
config['model']['arch_kwargs']['supervised'] = False
config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794,    
                            "signal/speaker_int": 433} 

# add task loss params to hparas 
config['hparas']['task_loss_params'] = {
    "signal/word_int":
        {"loss_type": 'crossentropyloss',
        "weight": 1.0},                                       # init loss is ~200 
    "signal/speaker_int":
        {"loss_type": 'crossentropyloss',
        "weight": 1.0}
            }

config['model']['whisper_model'] = 'large-v3-turbo'


In [4]:
importlib.reload(whisper_module)

module = whisper_module.WhisperTransferModule(config=config)


In [5]:
trainer = L.Trainer(devices=1)
trainer.fit(module)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/pytorch/loops/utilities.py:72: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmu

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
outputs = trainer.predict(module, module.val_dataloader(), return_predictions=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [ ]:
top1_word = []
top1_speaker = []
top5_word = []
top5_speaker = []

for record in outputs:
    top1_word.append(record['top1']['signal/word_int'])
    top1_speaker.append(record['top1']['signal/speaker_int'])
    top5_word.append(record['top5']['signal/word_int'])
    top5_speaker.append(record['top5']['signal/speaker_int'])

In [ ]:
n_examples = len(outputs)
output_dict = {
    "word_top1_mean": torch.stack(top1_word).mean(),
    "word_top1_sem": torch.stack(top1_word).std() / np.sqrt(n_examples),
    "speaker_top1_mean": torch.stack(top1_speaker).mean(),
    "speaker_top1_sem": torch.stack(top1_speaker).std() / np.sqrt(n_examples),

    "word_top5_mean": torch.stack(top5_word).mean(),
    "word_top5_sem": torch.stack(top5_word).std() / np.sqrt(n_examples),
    "speaker_top5_mean": torch.stack(top5_speaker).mean(),
    "speaker_top5_sem": torch.stack(top5_speaker).std() / np.sqrt(n_examples),
}
output_dict = {key:val.item() for key,val in output_dict.items()}

In [ ]:
output_dict

{'word_top1_mean': 0.0341125950217247,
 'word_top1_sem': 0.0013719532871618867,
 'speaker_top1_mean': 0.24576574563980103,
 'speaker_top1_sem': 0.0032539963722229004,
 'word_top5_mean': 0.5456822514533997,
 'word_top5_sem': 0.00541748246178031,
 'speaker_top5_mean': 0.7976502776145935,
 'speaker_top5_sem': 0.0033717849291861057}

In [ ]:
output_dict = {}
for key, task_dict in outputs.items():
    for task, scores in task_dict.items():
        n_examples = len(scores)
        task_str = task.split('/')[-1]
        output_dict[f"{task_str}_{key}_mean"] = torch.cat(scores).mean()
        output_dict[f"{task_str}_{key}_sem"] = torch.cat(scores).std() / np.sqrt(n_examples)

AttributeError: 'list' object has no attribute 'items'

In [ ]:
output_vals = torch.cat([output['accuracy'] for output in outputs])
len(output_vals)

KeyError: 'accuracy'

In [ ]:
output_vals.mean()

tensor(0.0010)

In [ ]:

output_vals.std(unbiased=True) / (output_vals.size(0) ** 0.5)

tensor(0.0002)

In [ ]:
import pickle
with open('eval_jsin_results/ssl_barlow_word_resnet50_hparam_set_0_linear_eval_jsin.pkl', 'rb') as handle:

    results = pickle.load(handle)

In [ ]:
results

{'mean_acc': tensor(0.0018),
 'std_acc': tensor(0.0423),
 'sem_acc': tensor(0.0003)}